In [19]:

!pip install gymnasium stable-baselines3 numpy

In [20]:
PASS_LIST = [
    "mem2reg",
    "instcombine",
    "gvn",
    "licm",
    "simplifycfg"
]

In [21]:
import subprocess

def compile_to_bc(c_file, bc_file):
    subprocess.run(
        ["clang", "-O0", "-emit-llvm", "-c", c_file, "-o", bc_file],
        check=True
    )

In [22]:
def apply_pass(input_bc, output_bc, opt_pass):
    subprocess.run(
        ["opt", f"-{opt_pass}", input_bc, "-o", output_bc],
        check=True
    )


In [23]:
def bc_to_ir(bc_file, ir_file):
    subprocess.run(
        [r"C:\Program Files\LLVM\bin\llvm-dis.exe", bc_file, "-o", ir_file],
        check=True
    )


In [24]:
def extract_features(ir_file):
    with open(ir_file, "r") as f:
        lines = f.readlines()

    instr = sum(1 for l in lines if l.strip().startswith("%"))
    blocks = sum(1 for l in lines if l.strip().endswith(":"))
    funcs = sum(1 for l in lines if l.startswith("define"))

    return [instr, blocks, funcs, len(lines)]


In [25]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np

class LLVMOptEnv(gym.Env):
    def __init__(self, initial_features):
        self.action_space = spaces.Discrete(len(PASS_LIST))
        self.observation_space = spaces.Box(
            low=0, high=1e6, shape=(4,), dtype=np.int32
        )
        self.state = np.array(initial_features)

    def reset(self, seed=None, options=None):
        return self.state, {}

    def step(self, action):
        reward = np.random.randint(-10, 20)
        self.state = self.state + np.random.randint(-5, 5, size=4)
        done = False
        return self.state, reward, done, False, {}

In [26]:
env = LLVMOptEnv([1200, 45, 8, 900])

for episode in range(5):
    state, _ = env.reset()
    total_reward = 0
    for step in range(3):
        action = env.action_space.sample()
        state, reward, done, _, _ = env.step(action)
        total_reward += reward
    print(f"Episode {episode}: Total Reward = {total_reward}")

Episode 0: Total Reward = 3
Episode 1: Total Reward = 26
Episode 2: Total Reward = 39
Episode 3: Total Reward = 27
Episode 4: Total Reward = 18
